In [ ]:
# Level 4 : Structured Tables and Aggregations

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("TrainSchedulePipeline") \
.master("local[*]") \
.getOrCreate()

In [5]:
import pandas as pd

df_pandas = pd.read_csv("/home/jovyan/work/transformed_data.csv")

df_pandas.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance,journey_duration_min
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,2026-05-21 00:00:00,2026-05-21 10:25:00,0,105.0
1,2,107,THVM,260,228,196,164,THIVIM,1,2026-05-21 11:06:00,2026-05-21 11:08:00,32,105.0
2,3,107,KRMI,345,296,247,198,KARMALI,1,2026-05-21 11:28:00,2026-05-21 11:30:00,49,105.0
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,2026-05-21 12:10:00,2026-05-21 00:00:00,78,105.0
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,2026-05-21 00:00:00,2026-05-21 20:30:00,0,115.0


In [9]:
df_spark = spark.read.csv(
    "/home/jovyan/work/transformed_data.csv",
    header = True,
    inferSchema = True
)
df_spark.show(5)

+---+--------+------------+---+---+---+---+------------+------------+-------------------+-------------------+--------+--------------------+
| SN|Train_No|Station_Code| 1A| 2A| 3A| SL|Station_Name|Route_Number|       Arrival_time|     Departure_Time|Distance|journey_duration_min|
+---+--------+------------+---+---+---+---+------------+------------+-------------------+-------------------+--------+--------------------+
|  1|     107|         SWV|100|100|100|100|SAWANTWADI R|           1|2026-05-21 00:00:00|2026-05-21 10:25:00|       0|               105.0|
|  2|     107|        THVM|260|228|196|164|      THIVIM|           1|2026-05-21 11:06:00|2026-05-21 11:08:00|      32|               105.0|
|  3|     107|        KRMI|345|296|247|198|     KARMALI|           1|2026-05-21 11:28:00|2026-05-21 11:30:00|      49|               105.0|
|  4|     107|         MAO|490|412|334|256| MADGOAN JN.|           1|2026-05-21 12:10:00|2026-05-21 00:00:00|      78|               105.0|
|  1|     108|      

In [13]:
# Task 4.1 - Generate a train level table showing number of stops

stops_table = df_pandas.groupby('Train_No')['Station_Name']\
.count() \
.reset_index()

stops_table.columns = ['Train_No','Number_of_Stops']

stops_table = stops_table.sort_values(
    'Number_of_Stops',
    ascending = False 
).reset_index(drop = True)

print("Train-level Stops Table Created")
print("\nTop 10 Trains by Number of Stops:")
print(stops_table.head(10))
print("\nTotal Trains:", len(stops_table))

Train-level Stops Table Created

Top 10 Trains by Number of Stops:
   Train_No  Number_of_Stops
0     53041              118
1     13007              112
2     13049              111
3     58112              109
4     13008              108
5     53042              107
6     13050              106
7     58111              102
8     19019               98
9     19024               97

Total Trains: 11113


In [15]:
# Task 4.2 - Generate a train-level table showing total distance traveled

distance_table = df_pandas.groupby('Train_No')['Distance'] \
.max() \
.reset_index() 

distance_table.columns = ['Train_No','Total_distance_KM']

distance_table = distance_table.sort_values(
    'Total_distance_KM',
    ascending = False
).reset_index(drop = True)

print("Train-level Distance Table Created")
print("\nTop 10 Trains by Total Distance:")
print(distance_table.head(10))
print("\nTotal Trains:", len(distance_table))

Train-level Distance Table Created

Top 10 Trains by Total Distance:
   Train_No  Total_distance_KM
0     15905               4260
1     15906               4256
2      2515               3939
3     12507               3932
4     12515               3932
5     12516               3930
6     12508               3930
7     16317               3769
8     16318               3765
9     16687               3663

Total Trains: 11113


In [17]:
# Task 4.3 - Create a cross table comparing trains and stations

cross_table = pd.crosstab(
    df_pandas['Train_No'],
    df_pandas['Station_Code']
)

print("Cross Table Created")
print("Shape:", cross_table.shape)
print("\nSample Cross Table (5 trains x 5 stations):")
print(cross_table.iloc[:5, :5])

Cross Table Created
Shape: (11113, 8147)

Sample Cross Table (5 trains x 5 stations):
Station_Code  AABH  AADR  AAG  AAH  AAL
Train_No                               
107              0     0    0    0    0
108              0     0    0    0    0
128              0     0    0    0    0
290              0     0    0    0    0
401              0     0    0    0    0


In [19]:
# Task 4.4 - Export all structured tables for reporting use

stops_table.to_csv(
    "/home/jovyan/work/stops_table.csv",
    index = False
)

distance_table.to_csv(
    "/home/jovyan/work/distance_table.csv",
    index = False
)

cross_table.to_csv(
    "/home/jovyan/work/cross_table.csv"
)

print("All Tables Exported Successfully ")

All Tables Exported Successfully 
